<a href="https://colab.research.google.com/github/Mariagiusi23/ID-001-AWOL-for-Audio/blob/main/notebook/03_Conditional_RealNVP_with_Learnable_Masks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Notebook 3 —  Conditional RealNVP with Learnable Masks

In this notebook we implement the **core AWOL model** for our audio setting.
I train a **conditional RealNVP flow** with learnable binary masks, where the
conditioning signal is provided by CLAP text embeddings. This flow defines a
probabilistic mapping from synthesizer parameters to a Gaussian latent space.

To enable deterministic inference, we also train a lightweight regressor
**g(c) → z**, which maps CLAP embeddings directly into the flow latent space.
At test time, a new prompt embedding can be mapped to parameters as:



#**🎯 STEP 1 — Setup**

**📍1.1 — Imports & global config**

In [ ]:
import os, io, json, math, random
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Optional: CLAP (Hugging Face). If you have precomputed embeddings, you can skip it.
# !pip -q install transformers torchaudio --upgrade
try:
    from transformers import AutoProcessor, ClapModel
    _HAS_CLAP = True
except Exception:
    _HAS_CLAP = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEVICE = {DEVICE}")


**📍1.2 — Small utilities**

In [ ]:
def safe_sigmoid(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    return torch.clamp(torch.sigmoid(x), eps, 1.0 - eps)

def l2_normalize(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    n = torch.linalg.norm(x, dim=-1, keepdim=True) + eps
    return x / n

def project_01(y: torch.Tensor) -> torch.Tensor:
    return torch.clamp(y, 0.0, 1.0)

def set_seed(seed: int = 42):
    import random, numpy as np, torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


#**🎯 STEP 2 — Data**

**📍2.1 — Inline dataset**

In [ ]:
TRAIN_ITEMS = [
    {"prompt": "a high whistle",               "params": [0.60, 0.90, 0.10, 0.30]},
    {"prompt": "a deep rumbling drone",        "params": [0.50, 0.05, 0.20, 0.70]},
    {"prompt": "a bright metallic ping",       "params": [0.70, 0.70, 0.80, 0.40]},
    {"prompt": "a mellow flute tone",          "params": [0.40, 0.40, 0.25, 0.20]},
    {"prompt": "an aggressive buzzy lead",     "params": [0.85, 0.65, 0.90, 0.90]},
    {"prompt": "a soft sine beep",             "params": [0.30, 0.30, 0.10, 0.05]},
    {"prompt": "a glassy bell",                "params": [0.55, 0.80, 0.60, 0.50]},
    {"prompt": "a warm pad",                   "params": [0.45, 0.35, 0.35, 0.25]},
    {"prompt": "a harsh resonant tone",        "params": [0.80, 0.50, 0.95, 0.75]},
    {"prompt": "a soft low hum",               "params": [0.25, 0.15, 0.20, 0.10]},
]

**📍2.2 — Convert to tensors**

In [ ]:
prompts = [row["prompt"] for row in TRAIN_ITEMS]
Y = torch.tensor([row["params"] for row in TRAIN_ITEMS], dtype=torch.float32)
N, K = Y.shape
print("N =", N, "| K =", K)

**📍2.3 — Train/Val split**

In [ ]:
set_seed(SEED)
idx = np.arange(N)
np.random.shuffle(idx)

val_frac = 0.25 if N >= 8 else 1
split = int(round(N * val_frac))

val_idx = idx[:split].tolist()
tr_idx  = idx[split:].tolist()

print(f"train={len(tr_idx)} / val={len(val_idx)}")

Y_tr = Y[tr_idx].to(DEVICE)
Y_val = Y[val_idx].to(DEVICE) if len(val_idx) > 0 else None
prompts_tr = [prompts[i] for i in tr_idx]
prompts_val = [prompts[i] for i in val_idx]


**📍2.4 — CLAP embeddings**

In [ ]:
if not _HAS_CLAP:
    torch.manual_seed(0)
    C = l2_normalize(torch.randn(N, 512))
    print("Using RANDOM unit-norm embeddings.")
else:
    ckpt = "laion/clap-htsat-unfused"
    processor = AutoProcessor.from_pretrained(ckpt)
    clap = ClapModel.from_pretrained(ckpt).to(DEVICE).eval()
    with torch.inference_mode():
        inputs = processor(text=prompts, return_tensors="pt", padding=True).to(DEVICE)
        feats = clap.get_text_features(**inputs)
        C = l2_normalize(feats).cpu()
    print("Computed CLAP embeddings:", tuple(C.shape))

C_tr = C[tr_idx].to(DEVICE)
C_val = C[val_idx].to(DEVICE) if len(val_idx) > 0 else None
C_DIM = C.shape[1]
print("C_DIM =", C_DIM)


#**🎯 STEP 3 — Conditional RealNVP**

**📍3.1 — Define conditional RealNVP with learnable masks**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def safe_sigmoid(x, eps=1e-6):
    return torch.clamp(torch.sigmoid(x), eps, 1. - eps)

class STBinary(torch.autograd.Function):
    @staticmethod
    def forward(ctx, p, thr: float):
        return (p > thr).float()
    @staticmethod
    def backward(ctx, g):
        return g, None

def binarize_st(p, thr=0.5):
    return STBinary.apply(p, thr)

class LearnableMask(nn.Module):
    """Binary mask with straight-through binarization; temperature τ sharpens probabilities."""
    def __init__(self, k: int, init_prob: float = 0.5):
        super().__init__()
        p0 = torch.full((k,), float(init_prob)).clamp(1e-3, 1-1e-3)
        logits = torch.log(p0) - torch.log(1 - p0)
        self.logits = nn.Parameter(logits)

    def forward(self, tau: float = 1.0, hard: bool = True):
        probs = safe_sigmoid(self.logits / tau)
        if hard:
            m = binarize_st(probs, 0.5)
        else:
            m = probs
        return m, probs

    def regularizer(self, target_frac=0.5):
        p = safe_sigmoid(self.logits)
        frac = p.mean()
        ent = - (p*torch.log(p+1e-8) + (1-p)*torch.log(1-p+1e-8)).mean()
        return (frac - target_frac).abs() + 0.1 * ent

class STNet(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, 2*out_dim)
        )
    def forward(self, x):
        s, t = self.net(x).chunk(2, dim=-1)
        s = torch.tanh(s)
        return s, t

class AffineCoupling(nn.Module):
    """Transform only the (1 - m) subset; conditioner input is [y*m, c]."""
    def __init__(self, k: int, c_dim: int, hidden: int = 128):
        super().__init__()
        self.mask = LearnableMask(k)
        self.st = STNet(k + c_dim, k, hidden)

    def forward(self, y, c, tau: float = 1.0):
        m, _ = self.mask(tau=tau, hard=True)
        m = m[None, :]
        yA = y * m
        s, t = self.st(torch.cat([yA, c], dim=-1))
        B = 1.0 - m
        zB = (y * B) * torch.exp(s * B) + t * B
        z = yA + zB
        logdet = (s * B).sum(dim=-1)
        return z, logdet

    def inverse(self, z, c, tau: float = 1.0):
        m, _ = self.mask(tau=tau, hard=True)
        m = m[None, :]
        zA = z * m
        s, t = self.st(torch.cat([zA, c], dim=-1))
        B = 1.0 - m
        yB = (z * B - t * B) * torch.exp(-s * B)
        y = zA + yB
        return y

    def mask_reg(self, target_frac=0.5):
        return self.mask.regularizer(target_frac=target_frac)

class RealNVPCond(nn.Module):
    def __init__(self, k: int, c_dim: int, n_layers: int = 4, hidden: int = 128):
        super().__init__()
        self.layers = nn.ModuleList([AffineCoupling(k, c_dim, hidden) for _ in range(n_layers)])
        self.k = k
        self.c_dim = c_dim

    def forward(self, y, c, tau: float = 1.0):
        x, logdet = y, 0.0
        for layer in self.layers:
            x, ld = layer(x, c, tau=tau)
            logdet = logdet + ld
        return x, logdet

    def inverse(self, z, c, tau: float = 1.0):
        x = z
        for layer in reversed(self.layers):
            x = layer.inverse(x, c, tau=tau)
        return x

    def mask_reg(self, target_frac=0.5):
        return sum([l.mask_reg(target_frac) for l in self.layers])


**📍3.2 — Train the conditional flow with NLL + mask regularization**

In [ ]:
from dataclasses import dataclass
import math, numpy as np

@dataclass
class FlowCfg:
    n_layers: int = 4
    hidden: int = 96
    lr: float = 3e-4
    weight_decay: float = 1e-4
    epochs: int = 1200
    grad_clip: float = 0.5
    tau_start: float = 2.0
    tau_end: float = 0.6
    mask_reg_w: float = 1e-2
    target_mask_frac: float = 0.5
    report_every: int = 100
    patience: int = 200

cfgF = FlowCfg()
flow = RealNVPCond(k=K, c_dim=C_DIM, n_layers=cfgF.n_layers, hidden=cfgF.hidden).to(DEVICE)
optF = torch.optim.AdamW(flow.parameters(), lr=cfgF.lr, weight_decay=cfgF.weight_decay)

def nll_std_gauss(z):
    return 0.5 * (z**2).sum(dim=-1)

best_val = float('inf'); best_sd = None; bad = 0
for ep in range(1, cfgF.epochs+1):
    t = (ep-1)/max(1, cfgF.epochs-1)
    tau = cfgF.tau_start*(1-t) + cfgF.tau_end*t

    flow.train(); optF.zero_grad()
    z, logdet = flow(Y_tr, C_tr, tau=tau)
    loss = (nll_std_gauss(z) - logdet).mean() + cfgF.mask_reg_w * flow.mask_reg(cfgF.target_mask_frac)
    loss.backward()
    if cfgF.grad_clip: nn.utils.clip_grad_norm_(flow.parameters(), cfgF.grad_clip)
    optF.step()

    val_loss = None
    if (Y_val is not None) and (C_val is not None) and len(Y_val) > 0:
        flow.eval()
        with torch.no_grad():
            zv, ldv = flow(Y_val, C_val, tau=tau)
            val_loss = (nll_std_gauss(zv) - ldv).mean().item()

    if ep % cfgF.report_every == 0 or ep == 1 or (val_loss is not None and val_loss < best_val):
        with torch.no_grad():
            fracs = [float(l.mask(tau=tau)[0].mean().item()) for l in flow.layers]
        msg = f"[{ep:04d}] tau={tau:.2f} train={loss.item():.4f}"
        if val_loss is not None: msg += f" | val={val_loss:.4f}"
        msg += f" | mask_on_frac={np.round(fracs,3)}"
        print(msg)

    if val_loss is not None:
        if val_loss < best_val - 1e-6:
            best_val = val_loss; best_sd = {k:v.detach().cpu().clone() for k,v in flow.state_dict().items()}; bad = 0
        else:
            bad += 1
            if bad >= cfgF.patience:
                print(f"⏹️ Early stop at epoch {ep} (best val={best_val:.4f})")
                break

if best_sd is not None:
    flow.load_state_dict(best_sd)
print("✅ Flow trained.")


**📍3.3 — Train g(c)→z end-to-end against y**

In [ ]:
class MLP_Z(nn.Module):
    def __init__(self, c_dim: int, k: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(c_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, k),
        )
    def forward(self, c):
        return self.net(c)

g = MLP_Z(C_DIM, K, hidden=256).to(DEVICE)

@dataclass
class GCfg:
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 3000
    report_every: int = 200
    patience: int = 400
    tau_infer: float = 0.6
cfgG = GCfg()

optG = torch.optim.AdamW(g.parameters(), lr=cfgG.lr, weight_decay=cfgG.weight_decay)

def rmse(a,b): return torch.sqrt(((a-b)**2).mean()).item()

best_val = float('inf'); best_sd = None; bad = 0
for ep in range(1, cfgG.epochs+1):
    g.train(); flow.eval()
    optG.zero_grad()
    y_hat_tr = torch.clamp(flow.inverse(g(C_tr), C_tr, tau=cfgG.tau_infer), 0.0, 1.0)
    loss_tr = F.mse_loss(y_hat_tr, Y_tr)
    loss_tr.backward()
    optG.step()

    val_report = ""
    if (Y_val is not None) and (C_val is not None) and len(Y_val) > 0:
        g.eval(); flow.eval()
        with torch.no_grad():
            y_hat_val = torch.clamp(flow.inverse(g(C_val), C_val, tau=cfgG.tau_infer), 0.0, 1.0)
            val_rmse = rmse(y_hat_val, Y_val)
        if val_rmse < best_val - 1e-6:
            best_val = val_rmse; best_sd = {k:v.detach().cpu().clone() for k,v in g.state_dict().items()}; bad = 0
        else:
            bad += 1
            if bad >= cfgG.patience:
                print(f"⏹️ g early stop at {ep} (best val_RMSE={best_val:.4f})")
                break
        val_report = f" | val_RMSE(y)={val_rmse:.4f}"

    if ep % cfgG.report_every == 0 or ep == 1:
        with torch.no_grad():
            tr_rmse = rmse(y_hat_tr, Y_tr)
        print(f"[g {ep:04d}] MSE(y)={loss_tr.item():.6f} | tr_RMSE(y)={tr_rmse:.4f}{val_report}")

if best_sd is not None:
    g.load_state_dict(best_sd)

flow.eval(); g.eval()
with torch.no_grad():
    y_hat_tr = torch.clamp(flow.inverse(g(C_tr), C_tr, tau=cfgG.tau_infer), 0.0, 1.0)
tr_rmse = rmse(y_hat_tr, Y_tr)
print(f"Final TRAIN RMSE(y): {tr_rmse:.4f}")
if (Y_val is not None) and (C_val is not None) and len(Y_val) > 0:
    with torch.no_grad():
        y_hat_val = torch.clamp(flow.inverse(g(C_val), C_val, tau=cfgG.tau_infer), 0.0, 1.0)
    val_rmse = rmse(y_hat_val, Y_val)
    print(f"Final VAL RMSE(y):   {val_rmse:.4f}")


📍**3.4 — Save checkpoint**

In [ ]:
import os, json

OUTDIR = "outputs_nb03"
os.makedirs(OUTDIR, exist_ok=True)

torch.save({
    "state_dict": flow.state_dict(),
    "k": K, "c_dim": C_DIM,
    "n_layers": len(flow.layers),
    "hidden": flow.layers[0].st.net[0].out_features if len(flow.layers)>0 else 128,
}, os.path.join(OUTDIR, "realnvp_cond_mask.pt"))

torch.save({
    "state_dict": g.state_dict(),
    "c_dim": C_DIM, "k": K
}, os.path.join(OUTDIR, "mlp_z.pt"))

print("✅ Saved flow and g() to", OUTDIR)

@torch.no_grad()
def predict_params(flow_model, g_model, C_vec, mode="mlp", tau=0.6):
    if mode == "mlp":
        z = g_model(C_vec)
    elif mode == "zero":
        z = torch.zeros((C_vec.shape[0], flow_model.k), device=C_vec.device)
    elif mode == "sample":
        z = torch.randn((C_vec.shape[0], flow_model.k), device=C_vec.device)
    else:
        raise ValueError("mode ∈ {'mlp','zero','sample'}")
    y = torch.clamp(flow_model.inverse(z, C_vec, tau=tau), 0.0, 1.0)
    return y


# **📊Results obtained**

In [ ]:
# Define the keys for the parameters
PARAM_KEYS = [f"param_{i+1}" for i in range(K)]
print("PARAM_KEYS defined:", PARAM_KEYS)

In [ ]:
import pandas as pd

df = pd.DataFrame([{"prompt": p, **{PARAM_KEYS[i]: row[i] for i in range(K)}}
                   for p, row in zip(prompts, Y.cpu().numpy())])
df

In [ ]:
# Save the DataFrame to a CSV file
df.to_csv('training_data.csv', index=False)
print("DataFrame df saved to training_data.csv")

# **🎯Plots**

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    y_hat_val = predict_params(flow, g, C_val, mode="mlp") if C_val is not None else None

if y_hat_val is not None:
    fig, axes = plt.subplots(1, K, figsize=(4*K, 4))
    if K == 1: axes = [axes]
    for d in range(K):
        axes[d].scatter(Y_val[:, d].cpu(), y_hat_val[:, d].cpu(), s=20, alpha=0.7)
        axes[d].plot([0,1],[0,1], 'k--')
        axes[d].set_title(f"{PARAM_KEYS[d]} (VAL)")
        axes[d].set_xlabel("True")
        axes[d].set_ylabel("Pred")
    plt.tight_layout()
    plt.show()


In [ ]:
if y_hat_val is not None:
    resid = (y_hat_val - Y_val).cpu().numpy().reshape(-1)
    plt.hist(resid, bins=20, alpha=0.7)
    plt.title("Residuals on VAL (all params)")
    plt.xlabel("Residual")
    plt.ylabel("Count")
    plt.show()


# ✅ Conclusions (Notebook 3 — Conditional RealNVP with Learnable Masks
)

In this notebook I successfully trained the conditional RealNVP with
learnable masks and the auxiliary regressor g(c)→z. The flow showed stable
training dynamics, while the regressor achieved low reconstruction error
(RMSE) on both train and validation sets.

Key takeaways:

- Binary masks converged to meaningful ON-fractions, avoiding collapse.
- The flow is invertible and reconstructs training parameters with low error.
- The regressor g enables fast, deterministic inference from CLAP embeddings.

This completes the **core AWOL mapping stage**.  
In the next notebook (04) I will use these trained models to explore
latent-space interpolations, test unseen prompts, and finally render audio
with the parametric synthesizer.
